In [ ]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname



root_path = dirname(os.getcwd()) + "/SEPH_TIME"

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = "cpu"

In [ ]:
with open("data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [ ]:
list(datasets_info.keys())

In [ ]:
dataset = "bpic2015_4"

In [ ]:
if dataset == "BPIC15_4_f2":
    with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]
elif dataset.startswith("BPIC15"):
    with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)["BPIC15_common"]
else:
    with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]

In [ ]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [ ]:
tab_all = pd.read_csv(data_dir_processed+dataset+"_processed_all.csv")
tab_all.head()

In [ ]:
import pandas as pd

tab_test = pd.read_csv(data_dir_processed + f"{dataset}_processed_test.csv")

# time:timestamp should be in float representing number of seconds.
durations = (
    tab_test
    .groupby("CaseID")["time:timestamp"]
    .agg(start="min", end="max")
)
durations["duration_sec"] = (durations["end"] - durations["start"])

# 4) Compute averages
avg_sec  = durations["duration_sec"].mean()
avg_days = avg_sec / (24 * 3600)

print(f"Average trace duration (test set): {avg_sec:,.0f} seconds\n")
print(f"which means ≃ {avg_days:.1f} days")


In [ ]:
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [ ]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "rb") as f:
    X_train = pickle.load(f)
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "rb") as f:
    X_valid = pickle.load(f)
#with open(data_dir_graphs + dataset + "_TEST4_repair.pkl", "rb") as f:
#    X_test = pickle.load(f)

max_len_test = tab_test.groupby("CaseID").size().max()
MaxPrefix = min(20,max_len_test-1)

X_tests = {}
for L in range(1, MaxPrefix + 1):
    fname = f"{dataset}_TEST{L}_repair.pkl"
    path  = os.path.join(data_dir_graphs, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Expected file not found: {path}")
    with open(path, "rb") as f:
        X_tests[L] = pickle.load(f)
    print(f"Loaded {len(X_tests[L])} graphs for prefix length {L} from {fname}")

In [ ]:

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected, NormalizeFeatures, Compose

#transform = ToUndirected()
transform = Compose([ToUndirected(), NormalizeFeatures()])

with torch.no_grad():
        for i in range(len(X_train)):
                X_train[i] = transform(X_train[i])
        for i in range(len(X_valid)):
                X_valid[i] = transform(X_valid[i])
        for L, graphs_L in X_tests.items():
                for i in range(len(graphs_L)):
                        graphs_L[i] = transform(graphs_L[i])
    


In [ ]:
edge_types = set()
node_types = set()
for i in range(len(X_train)):
    n, edge_type = X_train[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_valid)):
    n, edge_type = X_valid[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for graphs_L in X_tests.values():
    for i in range(len(graphs_L)):
        n, edge_type = graphs_L[i].metadata()
        for x in n:
            node_types.add(x)
        for x in edge_type:
            edge_types.add(x)




In [ ]:
node_types = list(node_types)
edge_types = list(edge_types)

In [ ]:
node_types

In [ ]:
edge_types

## Hyperopt

In [ ]:
from ax.service.managed_loop import optimize

In [ ]:
from torch_geometric.nn import (
    HeteroConv,
    global_mean_pool,
    GATv2Conv,
    SAGEConv,
    TransformerConv
)
from torch.nn import (
    ModuleList,
    Module,
    Linear
  )
from typing_extensions import Self

In [ ]:
from torch.nn import Module, ModuleList, Linear
import torch.nn as nn
from torch_geometric.nn import HeteroConv, SAGEConv, global_mean_pool

class HGNN(Module):
    def __init__(self, node_types, edge_types, parameters):
        super().__init__()
        hid = parameters["hid"]
        layers = parameters["layers"]
        aggregation = parameters["aggregation"]
        
        self.node_types = node_types
        self.edge_types = edge_types
        
        # Convolutional layers for heterogeneous graph
        self.convs = ModuleList()
        for _ in range(layers):
            conv = HeteroConv(
                {relation: SAGEConv((-1, -1), hid, aggr=aggregation)
                 for relation in edge_types},
                aggr=aggregation
            )
            self.convs.append(conv)
        
        # Linear layer for graph-level prediction
        self.lin = Linear(len(node_types) * hid, 1)
    
    def forward(self, batch):
        x_dict = batch.x_dict
        edge_index_dict = batch.edge_index_dict
        
        # Apply convolutional layers
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: x.relu() for key, x in x_dict.items()}
        
        # Pool node features for each node type
        graph_features = []
        for node_type in self.node_types:
            x = x_dict[node_type]
            batch_idx = batch[node_type].batch  # Batch index for pooling
            pooled = global_mean_pool(x, batch_idx)
            graph_features.append(pooled)
        
        # Concatenate pooled features
        graph_features = torch.cat(graph_features, dim=-1)
        
        # Predict remaining time
        output = self.lin(graph_features).squeeze(-1)
        return output  # Shape: [batch_size]

In [ ]:
from torcheval.metrics.functional import multiclass_accuracy, multiclass_f1_score
import torch.nn as nn
import time

In [ ]:
from torch_geometric.data import DataLoader
from copy import deepcopy

def train_hgnn(config, node_types, edge_types, epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = HGNN(node_types=node_types, edge_types=edge_types, parameters=config).to(device)
    loss_fn = nn.L1Loss()  # MAE for regression
    optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"])
    
    train_loader = DataLoader(X_train, batch_size=config["batch_size"], shuffle=True)
    valid_loader = DataLoader(X_valid, batch_size=config["batch_size"], shuffle=False)
    
    best_model = None
    best_loss = float("inf")
    patience = 5
    pat_count = 0
    
    for epoch in range(epochs):
        net.train()
        train_losses = []
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            preds = net(batch)  # Graph-level predictions
            true = batch.y  # Graph-level targets
            loss = loss_fn(preds, true)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        
        avg_train_loss = sum(train_losses) / len(train_losses)
        
        net.eval()
        valid_losses = []
        with torch.no_grad():
            for batch in valid_loader:
                batch = batch.to(device)
                preds = net(batch)
                true = batch.y
                valid_losses.append(loss_fn(preds, true).item())
        
        avg_val_loss = sum(valid_losses) / len(valid_losses)
        
        print(f"Epoch {epoch+1}/{epochs}, Train MAE: {avg_train_loss:.4f}, Valid MAE: {avg_val_loss:.4f}")
        
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            best_model = deepcopy(net)
            pat_count = 0
        else:
            pat_count += 1
            if pat_count >= patience:
                print("Early stopping")
                break
    
    return best_model

In [ ]:
trial_id = 0

def test_hgnn(net):
    global trial_id
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.eval()

    from sklearn.metrics import mean_absolute_error
    import numpy as np

    prefix_maes = []
    prefix_counts = []

    for L in range(1, MaxPrefix + 1):
        graphs = X_tests[L]
        if len(graphs) == 0:
            continue

        loader = DataLoader(graphs, batch_size=128, shuffle=False)
        y_true = []
        y_pred = []

        with torch.no_grad():
            for batch in loader:
                batch = batch.to(device)
                preds = net(batch)
                y_pred.extend(preds.cpu().numpy())
                y_true.extend(batch.y.cpu().numpy())

        mae = mean_absolute_error(y_true, y_pred)
        prefix_maes.append(mae / 86400)
        prefix_counts.append(len(y_true))

    maes = np.array(prefix_maes)
    counts = np.array(prefix_counts)
    total = counts.sum()

    weighted_mae = np.average(maes, weights=counts)
    weighted_var = np.average((maes - weighted_mae) ** 2, weights=counts)
    weighted_std = np.sqrt(weighted_var)

    if(trial_id !=0):
        trial_data = {"trial_id": trial_id}
    
        # Add MAE for each prefix length
        for L in range(1, MaxPrefix + 1):
            trial_data[f"prefix_{L}_mae"] = prefix_maes[L-1]
            trial_data[f"prefix_{L}_n"] = prefix_counts[L-1]

        # Save to CSV
        prefix_df = pd.DataFrame([trial_data])  # Single-row DataFrame
        prefix_df.to_csv(
            f"results/{dataset}_prefix.csv",
            mode="a",
            header=not os.path.exists(f"results/{dataset}_prefix.csv"),
            index=False,
        )

    
    trial_id += 1


    print(f"Weighted MAE (days): {weighted_mae:.4f} ± {weighted_std:.4f}")
    
    

    return {
        "remaining_time_mae": weighted_mae,
        "remaining_time_std": weighted_std,
    }

In [ ]:
outputreal = ["remaining_time"]
print(outputreal)

In [ ]:
def train_evaluate(config):
    trained_net = train_hgnn(config, node_types=node_types,edge_types=edge_types, epochs=50)
    return test_hgnn(trained_net)

In [ ]:
import logging

logging.getLogger("root").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:

sample_config = {
    "hid":          128, #128
    "layers":       2, #2
    "lr":           1e-3,
    "batch_size":   256, #256
    "aggregation": "mean",
}

# 1-epoch train just to exercise the code-path and see prints
net = train_hgnn(sample_config,node_types=node_types,edge_types=edge_types, epochs=1)

# run the test (with your debug prints enabled)
res = test_hgnn(net)
print("test_hgnn returned:", res)


In [ ]:
#Clear old prefix file
prefix_file = f"results/{dataset}_prefix.csv"
if os.path.exists(prefix_file):
    os.remove(prefix_file)

In [ ]:


from ax import Metric
tracking_metrics = [Metric(name="remaining_time_std")]

best_parameters, values, experiment, model = optimize(
    parameters=[
        {"name": "hid", "type": "choice", "values": [128], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "layers", "type": "choice", "values": [2, 3, 4, 5], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "layers", "type": "choice", "values": [2], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "lr", "type": "range", "bounds": [1e-4, 1e-1], "value_type": "float", "log_scale": True},
        {"name": "batch_size", "type": "choice", "values": [128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False}, 
        
        #{"name": "heads", "type": "choice", "values": [1,2], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "heads", "type": "choice", "values": [1], "value_type": "int", "is_ordered" : True,"sort_values":False},
        
        {"name": "aggregation", "type" : "choice", "values" :["sum", "mean", "max"], "value_type" : "str"}
        #{"name": "aggregation", "type" : "choice", "values" :["max"], "value_type" : "str"},
     
    ],
  
    evaluation_function=lambda config:train_evaluate({**config,"nodes_relations": edge_types}),
    objective_name='remaining_time_mae', 
    arms_per_trial=1,
    minimize = True,
    random_seed = 123,
    total_trials = 30
)

print("Best parameters:", best_parameters)
means, covariances = values
print("Means", means)
print("Experiment:", experiment)

In [ ]:
from ax.service.utils.report_utils import exp_to_df
results = exp_to_df(experiment)

In [ ]:
results = results.sort_values(by="remaining_time_mae")
results.to_csv(f"results/{dataset}.csv", sep=",")